In [ ]:
  !git clone https://github.com/justchenhao/BIT_CD.git
  %cd BIT_CD
  !pip install einops -q
  !touch datasets/__init__.py
  !sed -i 's|from torchvision.models.utils import load_state_dict_from_url|from torch.hub import load_state_dict_from_url|' models/resnet.py
  !sed -i 's/dtype=np.str)/dtype=str)/' datasets/CD_dataset.py
  !sed -i 's/map_location=self.device)/map_location=self.device, weights_only=False)/' models/basic_model.py models/evaluator.py
  print("repo ready + 4 fixes applied")


Cloning into 'BIT_CD'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (2/2), done.
remote: Total 92 (delta 1), reused 1 (delta 1), pack-reused 90 (from 1)
Receiving objects: 100% (92/92), 57.58 MiB | 14.85 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/BIT_CD
repo ready + 4 fixes applied


In [ ]:
  !wget -q --show-progress -O LEVIR-CD256.zip "https://www.dropbox.com/s/18fb5jo0npu5evm/LEVIR-CD256.zip?dl=1"
  !ls -lh LEVIR-CD256.zip
  !unzip -q LEVIR-CD256.zip -d ./data_levir && find ./data_levir -maxdepth 2 -type d


LEVIR-CD256.zip     100%[===================>]   2.31G  21.3MB/s    in 2m 35s  
-rw-r--r-- 1 root root 2.4G Jun 14 15:32 LEVIR-CD256.zip
./data_levir
./data_levir/LEVIR-CD256
./data_levir/LEVIR-CD256/label
./data_levir/LEVIR-CD256/B
./data_levir/LEVIR-CD256/list
./data_levir/LEVIR-CD256/A


In [ ]:
  !sed -i "s|path to the root of LEVIR-CD dataset|/content/BIT_CD/data_levir/LEVIR-CD256|" data_config.py


In [ ]:
  !grep -A1 "data_name == 'LEVIR'" data_config.py


        if data_name == 'LEVIR':
            self.root_dir = '/content/BIT_CD/data_levir/LEVIR-CD256'


In [ ]:
  !sh scripts/eval.sh

/content/BIT_CD/models/networks.py:266: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if with_pos is 'learned':
/content/BIT_CD/models/networks.py:297: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if self.pool_mode is 'max':
/content/BIT_CD/models/networks.py:299: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif self.pool_mode is 'ave':
True
[0]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106

In [ ]:
  !sed -i "s|path to the root of LEVIR-CD dataset|/content/BIT_CD/data_levir/LEVIR-CD256|" data_config.py
  !grep -A1 "LEVIR'" data_config.py


        if data_name == 'LEVIR':
            self.root_dir = '/content/BIT_CD/data_levir/LEVIR-CD256'
--
    data = DataConfig().get_data_config(data_name='LEVIR')
    print(data.data_name)


In [ ]:
  from google.colab import drive
  drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
  !mkdir -p /content/drive/MyDrive/bit_cd_ckpts
  !sed -i 's|checkpoint_root=checkpoints|checkpoint_root=/content/drive/MyDrive/bit_cd_ckpts|' scripts/run_cd.sh
  !grep checkpoint_root scripts/run_cd.sh


checkpoint_root=/content/drive/MyDrive/bit_cd_ckpts
python main_cd.py --img_size ${img_size} --checkpoint_root ${checkpoint_root} --lr_policy ${lr_policy} --split ${split} --split_val ${split_val} --net_G ${net_G} --gpu_ids ${gpus} --max_epochs ${max_epochs} --project_name ${project_name} --batch_size ${batch_size} --data_name ${data_name}  --lr ${lr}


In [ ]:

  !sed -i 's/map_location=self.device)/map_location=self.device, weights_only=False)/' models/trainer.py
  !grep -n "weights_only" models/trainer.py


104:                                    map_location=self.device, weights_only=False)


In [11]:
  !sh scripts/run_cd.sh

True
[0]
initialize network with normal
cuda:0
================ (Mon Jun 15 11:14:43 2026) ================
gpu_ids: [0] project_name: CD_base_transformer_pos_s4_dd8_LEVIR_b8_lr0.01_trainval_test_200_linear checkpoint_root: /content/drive/MyDrive/bit_cd_ckpts num_workers: 4 dataset: CDDataset data_name: LEVIR batch_size: 8 split: trainval split_val: test img_size: 256 n_class: 2 net_G: base_transformer_pos_s4_dd8 loss: ce optimizer: sgd lr: 0.01 max_epochs: 200 lr_policy: linear lr_decay_iters: 100 checkpoint_dir: /content/drive/MyDrive/bit_cd_ckpts/CD_base_transformer_pos_s4_dd8_LEVIR_b8_lr0.01_trainval_test_200_linear vis_dir: vis/CD_base_transformer_pos_s4_dd8_LEVIR_b8_lr0.01_trainval_test_200_linear loading last checkpoint...
Epoch_to_start = 84, Historical_best_acc = 0.9395 (at epoch 63)

lr: 0.0058209
Is_training: True. [84,199][1,1018], imps: 17.59, est: 119.35h, G_loss: 0.02067, running_mf1: 0.95557
Is_training: True. [84,199][101,1018], imps: 236.85, est: 8.86h, G_loss: 0.0104

In [11]:
-